In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
NUM_CITIES = int(input("Enter the number of cities: "))

city_names = []
for i in range(NUM_CITIES):
    name = input(f"Enter name for city {i+1}: ")
    city_names.append(name)

In [ ]:
def distance(city1, city2):
    return np.linalg.norm(city1 - city2)

def total_distance(route):
    dist = 0
    for i in range(len(route)):
        dist += distance(cities[route[i]], cities[route[(i+1) % len(route)]])
    return dist

In [ ]:
POP_SIZE = 100

def create_population():
    return [random.sample(range(NUM_CITIES), NUM_CITIES) for _ in range(POP_SIZE)]

def fitness(route):
    total_dist = total_distance(route)
    if total_dist == 0:
        return 0.0001
    return 1 / total_dist

def selection(population):
    valid_fitness = [fitness(r) for r in population]
    if all(f == 0 for f in valid_fitness):
        return random.choice(population)
    return random.choices(population, weights=valid_fitness)[0]

def crossover(p1, p2):
    start, end = sorted(random.sample(range(NUM_CITIES), 2))
    child = [-1] * NUM_CITIES
    child[start:end] = p1[start:end]
    ptr = 0
    for gene in p2:
        if gene not in child:
            while child[ptr] != -1:
                ptr += 1
            child[ptr] = gene
    return child

def mutate(route, rate=0.02):
    if random.random() < rate:
        i, j = random.sample(range(NUM_CITIES), 2)
        route[i], route[j] = route[j], route[i]
    return route

In [ ]:
def plot_route(route, title):
    ordered_cities = cities[route]
    ordered_cities = np.vstack([ordered_cities, ordered_cities[0]])
    plt.plot(ordered_cities[:, 0], ordered_cities[:, 1], 'o-')
    plt.title(title)
    for i, city_idx in enumerate(route):
        plt.annotate(city_names[city_idx], (cities[city_idx, 0], cities[city_idx, 1]), textcoords="offset points", xytext=(5, 5))
    plt.show()

In [ ]:
import googlemaps

GOOGLE_MAPS_API_KEY = 'YOUR_API_KEY'
gmaps = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)

def geocode_city(city_name):
    try:
        geocode_result = gmaps.geocode(city_name)
        if geocode_result:
            location = geocode_result[0]['geometry']['location']
            return [location['lat'], location['lng']]
        else:
            return [0, 0]
    except Exception as e:
        return [0, 0]

cities_coords = [geocode_city(name) for name in city_names]
cities = np.array(cities_coords)

print("Geocoded cities:")
for i, name in enumerate(city_names):
    print(f"- {name}: {cities[i]}")

In [ ]:
GENERATIONS = 500
population = create_population()

for gen in range(GENERATIONS):
    new_population = []
    for _ in range(POP_SIZE):
        p1 = selection(population)
        p2 = selection(population)
        child = crossover(p1, p2)
        child = mutate(child)
        new_population.append(child)
    population = new_population

best_route = max(population, key=fitness)
print(f"Best distance: {total_distance(best_route):.2f}")
print(f"Best route: {[city_names[i] for i in best_route]}")

plot_route(best_route, f"Best Route (Distance: {total_distance(best_route):.2f})")